In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from rnacappredictor.predict_cap import (
    generate_fingerprint_mixes,
    predict_cap,
    predict_cap_with_saved_model,
    create_features,
    save_knn_model,
    load_knn_model,
    mix_fingerprints
)

## Configuration
### Set paths and model parameters

In [2]:
# Configuration
TRAINING_DATA_PATH = "../data/FM179-FM181_fingerprints.csv"
TEST_DATA_PATH = "/media/gibran/Data/Data/FM219/no_sample_id/20251015_1646_MD-101425_FBD19838_4faf599b/fastq_pass/U6/fingerprints.csv"
MODEL_SAVE_PATH = "../models/knn_model_Vopalensky_test16.pkl"
FIGURES_PATH = "../figures/"

# Create directories if they don't exist
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)

print(f"Model path: {MODEL_SAVE_PATH}")
print(f"Model exists: {os.path.exists(MODEL_SAVE_PATH)}")

Model path: ../models/knn_model_Vopalensky_test16.pkl
Model exists: True


### Step 1: Load TRAINING data (FM179-FM181 with known cap types)

In [13]:
# Load TRAINING data - this has pure cap fingerprints with known 'cap' column
df_train = pd.read_csv(TRAINING_DATA_PATH)
df = pd.read_csv(TRAINING_DATA_PATH)

print("Training data loaded:")
print(df_train.head())
print(f"\nShape: {df_train.shape}")
print(f"\nCaps in training data:")
print(df_train['cap'].value_counts())
print(f"\nRTs in training data:")
print(df_train['RT'].unique())

Training data loaded:
     barcode      isoform  num_reads  num_A  num_C  num_G  num_T  num_DEL  \
0  barcode02  E_(5_6_7_8)      17157   6617    820   1833   3059     2068   
1  barcode06  E_(5_6_7_8)      18227   8620    903    586   6890      567   
2  barcode10  E_(5_6_7_8)      22001   7932    831   1314   9330      810   
3  barcode14  E_(5_6_7_8)      20662   3248    902    497  13492     1223   
4  barcode18  E_(5_6_7_8)      20681   9262   1114    790   7201      351   

   num_INS  num_reads_ACGT  ...  INS%_INSDEL  DEL%_INSDEL        A%        C%  \
0     2760           12329  ...     0.160867     0.120534  0.536702  0.066510   
1      661           16999  ...     0.036265     0.031108  0.507089  0.053121   
2     1784           19407  ...     0.081087     0.036817  0.408719  0.042820   
3     1300           18139  ...     0.062917     0.059191  0.179062  0.049727   
4     1963           18367  ...     0.094918     0.016972  0.504274  0.060652   

         G%        T%  barco

### Saving two different libraries
#### 1. exclude_zero_caps=True
####   Use when you want to force the model to consider mixtures where all selected caps are present.

#### 2. exclude_zero_caps=False
####   Use when you want to allow caps to be absent, useful for blind-test style exploration.

In [12]:
MIXES_TRUE_PATH = "../models/df_train_mixes_exclude_zero_TRUE_step002.parquet"
FEATURES_TRUE_PATH = "../models/training_features_exclude_zero_TRUE_step002.npz"

MIXES_TRUE_PATH_INSDEL = "../models/df_train_mixes_exclude_zero_TRUE_INSDEL_step002.parquet"
FEATURES_TRUE_PATH_INSDEL = "../models/training_features_exclude_zero_TRUE_INSDEL_step002.npz"

MIXES_FALSE_PATH = "../models/df_train_mixes_exclude_zero_FALSE_step002.parquet"
FEATURES_FALSE_PATH = "../models/training_features_exclude_zero_FALSE_step002.npz"

MIXES_FALSE_PATH_INSDEL = "../models/df_train_mixes_exclude_zero_FALSE_INSDEL_step002.parquet"
FEATURES_FALSE_PATH_INSDEL = "../models/training_features_exclude_zero_FALSE_INSDEL_step002.npz"


### Generate the `True` version

In [6]:
# Define which caps to include in the model
cap_order = sorted(df_train['cap'].unique())

df_train_mixes_true = generate_fingerprint_mixes(
    df_train,
    step_size=0.02,
    include_insdel=False,
    cap_order=cap_order,
    replicates_per_cap=5,
    exclude_zero_caps=True
)

df_train_mixes_true.to_parquet(MIXES_TRUE_PATH, index=False)

all_rt_names_true = df_train_mixes_true["RT"].unique()
X_train_true, y_train_true, caps_train_true, experiments_train_true = create_features(
    df_train_mixes_true,
    all_rt_names_true,
    False
)

np.savez_compressed(
    FEATURES_TRUE_PATH,
    X_train=X_train_true,
    y_train=y_train_true,
    caps_train=caps_train_true,
    experiments_train=experiments_train_true,
    all_rt_names=all_rt_names_true
)


Generating fingerprint mixtures
Caps in mixture: ['Ap₄A-U1', 'NAD-U1', 'TMG-U1', 'm⁷Gp₃A-U1', 'ppp-U1', 'y-meGTP-U1']
Step size: 0.02
Total combinations to generate: 1906884
⚠️  IMPORTANT: exclude_zero_caps=True
   Only generating TRUE MIXTURES (all caps present)
   This prevents catch-all bias (recommended!)



Generating combinations: 100%|█████| 1906884/1906884 [1:49:10<00:00, 291.09it/s]


### Generate the `False` version

In [7]:
df_train_mixes_false = generate_fingerprint_mixes(
    df_train,
    step_size=0.02,
    include_insdel=False,
    cap_order=cap_order,
    replicates_per_cap=5,
    exclude_zero_caps=False
)

df_train_mixes_false.to_parquet(MIXES_FALSE_PATH, index=False)

all_rt_names_false = df_train_mixes_false["RT"].unique()
X_train_false, y_train_false, caps_train_false, experiments_train_false = create_features(
    df_train_mixes_false,
    all_rt_names_false,
    False
)

np.savez_compressed(
    FEATURES_FALSE_PATH,
    X_train=X_train_false,
    y_train=y_train_false,
    caps_train=caps_train_false,
    experiments_train=experiments_train_false,
    all_rt_names=all_rt_names_false
)


Generating fingerprint mixtures
Caps in mixture: ['Ap₄A-U1', 'NAD-U1', 'TMG-U1', 'm⁷Gp₃A-U1', 'ppp-U1', 'y-meGTP-U1']
Step size: 0.02
Total combinations to generate: 3329692
   Note: Include all combinations (including single-cap)
   Set exclude_zero_caps=True to prevent catch-all bias



Generating combinations: 100%|█████| 3329692/3329692 [3:12:02<00:00, 288.98it/s]


### Generate the `False` version with INSDEL

In [16]:
df_train_mixes_false = generate_fingerprint_mixes(
    df_train,
    step_size=0.02,
    include_insdel=True,
    cap_order=cap_order,
    replicates_per_cap=5,
    exclude_zero_caps=False
)

df_train_mixes_false.to_parquet(MIXES_FALSE_PATH_INSDEL, index=False)

all_rt_names_false = df_train_mixes_false["RT"].unique()
X_train_false, y_train_false, caps_train_false, experiments_train_false = create_features(
    df_train_mixes_false,
    all_rt_names_false,
    include_insdel=True
)

np.savez_compressed(
    FEATURES_FALSE_PATH_INSDEL,
    X_train=X_train_false,
    y_train=y_train_false,
    caps_train=caps_train_false,
    experiments_train=experiments_train_false,
    all_rt_names=all_rt_names_false
)


Generating fingerprint mixtures
Caps in mixture: ['Ap₄A-U1', 'NAD-U1', 'TMG-U1', 'm⁷Gp₃A-U1', 'ppp-U1', 'y-meGTP-U1']
Step size: 0.02
Total combinations to generate: 3329692
   Note: Include all combinations (including single-cap)
   Set exclude_zero_caps=True to prevent catch-all bias



Generating combinations: 100%|█████████████████████████████████████████████████| 3329692/3329692 [3:26:36<00:00, 268.60it/s]


### Generate the `True` version with INSDEL

In [14]:
# Define which caps to include in the model
cap_order = sorted(df_train['cap'].unique())

df_train_mixes_true = generate_fingerprint_mixes(
    df_train,
    step_size=0.02,
    include_insdel=True,
    cap_order=cap_order,
    replicates_per_cap=5,
    exclude_zero_caps=True
)

df_train_mixes_true.to_parquet(MIXES_TRUE_PATH_INSDEL, index=False)

all_rt_names_true = df_train_mixes_true["RT"].unique()
X_train_true, y_train_true, caps_train_true, experiments_train_true = create_features(
    df_train_mixes_true,
    all_rt_names_true,
    include_insdel=True
)

np.savez_compressed(
    FEATURES_TRUE_PATH_INSDEL,
    X_train=X_train_true,
    y_train=y_train_true,
    caps_train=caps_train_true,
    experiments_train=experiments_train_true,
    all_rt_names=all_rt_names_true
)


Generating fingerprint mixtures
Caps in mixture: ['Ap₄A-U1', 'NAD-U1', 'TMG-U1', 'm⁷Gp₃A-U1', 'ppp-U1', 'y-meGTP-U1']
Step size: 0.02
Total combinations to generate: 1906884
⚠️  IMPORTANT: exclude_zero_caps=True
   Only generating TRUE MIXTURES (all caps present)
   This prevents catch-all bias (recommended!)



Generating combinations: 100%|█████████████████████████████████████████████████| 1906884/1906884 [1:57:14<00:00, 271.08it/s]


ValueError: Column A% not found in DataFrame

The error above was corrected in the following cell :)

In [15]:
df_train_mixes_true.to_parquet(MIXES_TRUE_PATH_INSDEL, index=False)

all_rt_names_true = df_train_mixes_true["RT"].unique()
X_train_true, y_train_true, caps_train_true, experiments_train_true = create_features(
    df_train_mixes_true,
    all_rt_names_true,
    include_insdel=True
)

np.savez_compressed(
    FEATURES_TRUE_PATH_INSDEL,
    X_train=X_train_true,
    y_train=y_train_true,
    caps_train=caps_train_true,
    experiments_train=experiments_train_true,
    all_rt_names=all_rt_names_true
)